# A hiring process simulator
# CV Generator (Agent 1): Generates CVs from job titles (in real life, this can be a list of PDFs to read)
# Hiring Manager (Agent 2): Tells the recruiter to find matching CVs for a job phrase
# Recruiter (Agent 3): Awaits a query from the hiring manager (Agent 2) and then returns best match CV byreading CVs created by Agent 1


In [ ]:
# TODOLIST
# Split into multiple modules for better organization. Currently, all code is in a single file, which can be overwhelming and hard to maintain. Consider breaking it down into smaller modules based on functionality, such as agent definitions, utility functions, and main execution logic.

from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
from email.message import EmailMessage
from IPython.display import Markdown, display
# Next two for using OpenRouter
from openai import AsyncOpenAI
from agents import Agent, OpenAIChatCompletionsModel
load_dotenv(override=True)


In [ ]:
# Email configuration
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

if EMAIL_ADDRESS:
    print("Email address is set to:", EMAIL_ADDRESS)
else:
    print("Email address is not set")

if EMAIL_SMTP_SERVER:
    print("SMTP server is set to:", EMAIL_SMTP_SERVER)
else:
    print("SMTP server is not set")

if EMAIL_APP_PASSWORD:
    print("App password is set")
else:
    print("App password is not set")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

In [4]:

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    print(f"Sending email to {EMAIL_ADDRESS} with subject: {subject}")
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [5]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        print(f"Sending email with subject: {subject} because USE_EMAIL is True")
        send_email(subject, text_body, html_body)
    else:
        # Hack to write to a local file instead of sending a push notification for testing
        # push(f"Subject: {subject}\n\n{text_body}")
        # write from, to, subject, and body to a file for testing
        print(f"Writing message to test_message.txt with subject: {subject} because USE_EMAIL is False")
        with open("test_message.txt", "w") as f:
            f.write(f"From: {EMAIL_ADDRESS}\n")
            f.write(f"To: {EMAIL_ADDRESS}\n")
            f.write(f"Subject: {subject}\n")
            f.write(f"Body:\n{text_body}\n")

In [6]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (and this is optional)")

openrouter_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODEL_NAME = "google/gemma-4-26b-a4b-it"  # any model slug OpenRouter supports


OpenRouter API Key exists and begins sk-or-


In [7]:
cv_generation_instruction = f"""
You are a helpful assistant that generates CVs for job applications.
You will receive a list of job titles. For each title, generate between 1-3 CVs in the following format:
first_name, last_name, 
education - less than 10 words
work_experience highlghting 2-3 fictitious roles - less than 100 words
skills relevant to the job title - less than 50 words.
The agent should return an array of CVs in JSON format with the following structure:
first_name, last_name, education, work_experience, skills. 
"""
cv_generator_agent = Agent(
    name="cv_generator_agent",
    instructions=cv_generation_instruction,
    model=OpenAIChatCompletionsModel(
        model=MODEL_NAME,
        openai_client=openrouter_client)
)

In [8]:

# Create an agent to simulate a recruiter who is reviewing CVs and sending emails to the hiring manager. The agent should be able to read CVs, evaluate them, and send emails to the hiring manager with recommendations.
recruiter_intro = """
You will receive a job title phrases from the hiring manager.

You task is to review all the CVs and sending the name of one top matching candidate to the hiring manager. 

If you do not find a good match, you should send an email to the hiring manager that you did not find a good match.

If the title phrase is not clear, you should send an email to the hiring manager for clarification without reviewing the CVs. 

You should not make assumptions about the job title. You should only review the CVs if you have a clear understanding of the job title phrase.

"""

recruiter_instructions = recruiter_intro + "Your email style is professional. You should highlight the candidate name and top 3 matching themes."

recruiter_agent = Agent(
    name="Recruiter_Agent",
    instructions=recruiter_instructions,
    model=OpenAIChatCompletionsModel(
        model=MODEL_NAME,
        openai_client=openrouter_client,
    ),
)


In [9]:
# Create an agent to simulate a hiring manager who is requesting CVs from the recruiter. 
# The agent should be able to provide a job description phrase and receive recommendations from the recruiter.

hiring_manager_intro = """
You are a hiring manager in a tech company who is looking for candidates for a position in large software development group.
You will provide a 2-3 words job description phrase to the recruiter.
Be concise and clear.

Example: "Senior Data Scientist", "scrum master", "frontend developer", "backend engineer", "full stack developer", "machine learning engineer", "data analyst", "product manager", "UX designer", "DevOps engineer", "cloud architect", "cybersecurity specialist", "AI researcher", "mobile app developer", "blockchain developer", "game developer", "embedded systems engineer", "network administrator", "IT support specialist".
"""

hiring_manager_instructions = hiring_manager_intro + "Your email style is professional and concise."

hiring_manager_agent = Agent(name="Hiring_Manager_Agent", 
                             instructions=hiring_manager_intro, 
                             model=OpenAIChatCompletionsModel(
                                     model=MODEL_NAME,
                                     openai_client=openrouter_client)
)

In [ ]:
# Generate all CVs
# job_titles = ["Senior Data Scientist"]

job_titles = ["Senior Data Scientist", "Scrum Master", "Frontend Developer", "Backend Engineer", 
              "Full Stack Developer", "Machine Learning Engineer", "Data Analyst", "Product Manager", 
              "UX Designer", "DevOps Engineer"]
cv_bank = await Runner.run(cv_generator_agent, 
                           input="Generate CVs for the following job titles: " + ", ".join(job_titles))

print("Generated CVs:")
display(Markdown(cv_bank.final_output))





In [15]:
# Write the generated CVs into an HTML file "simulated_CVs.html" for easier viewing under output directory
# Create the output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

with open("output/simulated_CVs.html", "w") as f:
    f.write(f"<html><body>{cv_bank.final_output}</body></html>")

In [ ]:
# Generate a job description phrase from the hiring manager
query_job_category = await Runner.run(hiring_manager_agent, 
                                      input="Please provide a job description phrase from " + ", ".join(job_titles))
print("Job description phrase from hiring manager:")
display(Markdown(query_job_category.final_output))


In [ ]:
# Run the recruiter agent with the job description phrase from the hiring manager
recruiter_response = await Runner.run(recruiter_agent, 
                                      input=query_job_category.final_output+" Here are the CVs: " + cv_bank.final_output)
print("Recommendations from recruiter:")
display(Markdown(recruiter_response.final_output))
send_message("Recruiter Recommendations", recruiter_response.final_output, 
             f"<html><body>{recruiter_response.final_output}</body></html>")

# Write the recruiter response to an HTML file for easier viewing
with open("recruiter_recommendations.html", "w") as f:
    f.write(f"<html><body>{recruiter_response.final_output}</body></html>")
